In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt # Görselleştirme kütüphanemiz

# 1. Device and Data Preparation
device = "cuda" if torch.cuda.is_available() else "cpu"
df = pd.read_csv('time_series_60min_singleindex.csv', parse_dates=['utc_timestamp'], index_col='utc_timestamp')
solar_data = df[['AT_solar_generation_actual']].dropna()

scaler = MinMaxScaler(feature_range=(-1, 1))
solar_data_scaled = scaler.fit_transform(solar_data.values)

def create_sequences(data, lookback):
    X, y = [], []
    for i in range(len(data) - lookback):
        X.append(data[i:(i + lookback)])
        y.append(data[i + lookback])
    return np.array(X), np.array(y)

lookback = 24 
X, y = create_sequences(solar_data_scaled, lookback)

train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

X_train_tensor = torch.from_numpy(X_train).float()
y_train_tensor = torch.from_numpy(y_train).float()
X_test_tensor = torch.from_numpy(X_test).float()
y_test_tensor = torch.from_numpy(y_test).float()

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=False)
X_test_device = X_test_tensor.to(device)

criterion = nn.MSELoss()
epochs = 20  # Modelin daha iyi öğrenmesi için eğitim döngüsünü artırdık

# 2. Improved Models (Dropout Added)
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.layer_dim = layer_dim
        self.hidden_dim = hidden_dim
        # Dropouts added
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

class GRUModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(GRUModel, self).__init__()
        self.layer_dim = layer_dim
        self.hidden_dim = hidden_dim
        self.gru = nn.GRU(input_dim, hidden_dim, layer_dim, batch_first=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.gru(x, h0)
        return self.fc(out[:, -1, :])

class BiLSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(BiLSTMModel, self).__init__()
        self.layer_dim = layer_dim
        self.hidden_dim = hidden_dim
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True, bidirectional=True, dropout=0.2)
        self.fc = nn.Linear(hidden_dim * 2, output_dim)

    def forward(self, x):
        h0 = torch.zeros(self.layer_dim * 2, x.size(0), self.hidden_dim).to(x.device)
        c0 = torch.zeros(self.layer_dim * 2, x.size(0), self.hidden_dim).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

def train_and_predict(model_class, name):
    print(f"--- {name} Modeli Eğitiliyor ({epochs} Epoch) ---")
    model = model_class(1, 32, 2, 1).to(device)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    model.train()
    for epoch in range(epochs):
        for bx, by in train_loader:
            bx, by = bx.to(device), by.to(device)
            loss = criterion(model(bx), by)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
    model.eval()
    with torch.no_grad():
        preds = model(X_test_device).cpu().numpy()
    return preds

# Training Models (This part may take some time as Epoch increases)
lstm_preds = train_and_predict(LSTMModel, "LSTM")
gru_preds = train_and_predict(GRUModel, "GRU")
bilstm_preds = train_and_predict(BiLSTMModel, "Bi-LSTM")

# 3. Converting to MW and Fixing Negatives
gercek_mw = scaler.inverse_transform(y_test_tensor.numpy())
lstm_mw = np.clip(scaler.inverse_transform(lstm_preds), 0, None)
gru_mw = np.clip(scaler.inverse_transform(gru_preds), 0, None)
bilstm_mw = np.clip(scaler.inverse_transform(bilstm_preds), 0, None)

# 4. Visualizing Data (Matplotlib)
plt.figure(figsize=(16, 6))

# We plot the first 100 hours (about 4 days) in the test set
plt.plot(gercek_mw[:100], label='Actual Production (MW)', color='black', linewidth=3)
plt.plot(lstm_mw[:100], label='LSTM Prediction', linestyle='--', alpha=0.8)
plt.plot(gru_mw[:100], label='GRU Estimate', linestyle='-.', alpha=0.8)
plt.plot(bilstm_mw[:100], label='Bi-LSTM Prediction', linestyle=':', linewidth=2, alpha=0.8)

plt.title('Austrian Solar Energy Production: Reality vs. Artificial Intelligence Predictions (First 100 Hours)', fontsize=14)
plt.xlabel('Time (Hour)', fontsize=12)
plt.ylabel('Energy Production (Megawatt)', fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()

print("\nModel training has been completed, graphics are being created...")
plt.show()